(SurfaceBulk-ADR)=
# Surface-Bulk ADR

We demonstrate here how to solve general systems of coupled ADR equations.

We consider as an example the coupled PDE

\begin{align*}
\partial_t u + \nabla\cdot(\vec{b}u - d \nabla u) + cu = f \\
\partial_t v + \nabla\cdot(\vec{b}v - d \nabla v) + cv = f 
\end{align*}

where
- $u$ is the species considered
- $\vec{b}$ is the advection field
- $d$ is the diffusion coefficient
- $c$ is the reaction coefficient
- $f$ is the right-hand side

The code gives the opportunity to solve ADR equations in the bulk, on surfaces and coupled bulk-surface equations. We will go through all the cases and show:
- How to setup the various problems
- How to test that the code is correct
- The functionality available 

In [1]:
from ngsolve import *
from ngsolve.webgui import Draw
from cosmos.solvers.base_problem import UnsteadyProblem
from cosmos.utils.generate_surface_meshes import generate_circle
import time
import numpy as np

mesh, _ = generate_circle(maxh=0.1)

dt = Parameter(0.05)
T = 10
t = Parameter(0)
simulation = UnsteadyProblem(mesh=mesh, t=t, dt = dt, T = T)

from cosmos.solvers.adr import ADRSolver
solver = ADRSolver()
simulation.attach_solver(solver)

D_A = 100
D_X = 1
D_B = 1
k_on = 1
k_off = 1


# C_A
neu_d1 = {'.*': CF((0,0))}
solver.AddSpecie(VorB = VOL,
                u0 = 1,
                diffusion = D_A,
                neu_d = neu_d1)

# N_X
solver.AddSpecie(VorB = BND,
                u0 = 1,
                diffusion = D_X)

# N_B
solver.AddSpecie(VorB = BND,
                u0 = 0,
                diffusion = D_B)

def f0(trial):
    return k_on*trial[0]*trial[1] - k_off*trial[2]
solver.AddNonlinearity(marker0 = 0, markers = [0, 1, 2], f = f0, VorB = BND)

def f1(trial):
    return k_on*trial[0]*trial[1] - k_off*trial[2]
solver.AddNonlinearity(marker0 = 1, markers = [0, 1, 2], f = f1, VorB = BND)


def f2(trial):
    return -k_on*trial[0]*trial[1] + k_off*trial[2]
solver.AddNonlinearity(marker0 = 2, markers = [0, 1, 2], f = f2, VorB = BND)

simulation.run()

solver.draw_solution()

print('The average value of C_A in the volume is:', Integrate(solver.get_solution()[0], mesh)/pi)
print('The average value of N_X in the volume is:', Integrate(solver.get_solution()[1], mesh, BND)/(2*pi))
print('The average value of N_B in the volume is:', Integrate(solver.get_solution()[2], mesh, BND)/(2*pi))


Running Simulation...: 100%|█████████████| 200/200 [00:05<00:00, 33.39it/s]


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

The average value of C_A in the volume is: 0.4131294469390181
The average value of N_X in the volume is: 0.7069922615204904
The average value of N_B in the volume is: 0.29257987091746873
